In [1]:
!pip install mplsoccer ipywidgets pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.9 MB/s eta 0:00:00


In [2]:
!git lfs install
!git clone https://github.com/SkillCorner/opendata.git
!cd opendata && git lfs pull

Git LFS initialized.
Cloning into 'opendata'...
remote: Enumerating objects: 524, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 524 (delta 57), reused 53 (delta 46), pack-reused 431 (from 1)
Receiving objects: 100% (524/524), 159.84 MiB | 9.02 MiB/s, done.
Resolving deltas: 100% (206/206), done.
Filtering content: 100% (10/10), 873.32 MiB | 44.30 MiB/s, done.


In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Arc, Circle, Rectangle
from scipy.ndimage import gaussian_filter
import ipywidgets as widgets
from IPython.display import display

In [4]:
MATCH_ID = "1886347"
BASE = Path(f"opendata/data/matches/{MATCH_ID}")

meta_path = BASE / f"{MATCH_ID}_match.json"
tracking_path = BASE / f"{MATCH_ID}_tracking_extrapolated.jsonl"

if not meta_path.exists():
    raise FileNotFoundError(f"Match metadata file not found: {meta_path}")
if not tracking_path.exists():
    raise FileNotFoundError(f"Tracking file not found: {tracking_path}")

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

frames = []
with open(tracking_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            frames.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Skipping bad JSON on line {line_num}: {e}")

if not frames:
    raise ValueError("No valid tracking frames were loaded.")

print(f"Loaded {len(frames):,} frames")
print("Example keys:", list(frames[0].keys()))

Loaded 59,061 frames
Example keys: ['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data']


In [5]:
pitch_length = float(meta.get("pitch_length", 105))
pitch_width = float(meta.get("pitch_width",  68))

home_team_id = meta["home_team"]["id"]
away_team_id = meta["away_team"]["id"]
home_team_name = meta["home_team"]["name"]
away_team_name = meta["away_team"]["name"]

team_lookup = {
    home_team_id: home_team_name,
    away_team_id: away_team_name,
}

FRAME_RATE = 10

print(f"Pitch: {pitch_length} × {pitch_width} m")
print(f"Home : {home_team_id} — {home_team_name}")
print(f"Away : {away_team_id} — {away_team_name}")

Pitch: 104.0 × 68.0 m
Home : 4177 — Auckland FC
Away : 1805 — Newcastle United Jets FC


In [6]:
player_lookup = {}

for p in meta.get("players", []):
    pid = p.get("id")
    if pid is None:
        continue
    try:
        pid = int(pid)
    except Exception:
        continue

    player_lookup[pid] = {
        "player_id": pid,
        "name"     : p.get("short_name", str(pid)),
        "team_id"  : p.get("team_id"),
        "position" : p.get("player_role", {}).get("acronym", ""),
    }

frame_id_to_idx = {fr.get("frame"): i for i, fr in enumerate(frames)}

print(f"Players in lookup : {len(player_lookup)}")
print(f"Frames indexed    : {len(frame_id_to_idx):,}")

Players in lookup : 36
Frames indexed    : 59,061


In [7]:
def safe_int(v):
    try:
        return int(v)
    except Exception:
        return None

def euclidean(x1, y1, x2, y2):
    return float(np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2))

def point_to_segment_distance(px, py, x1, y1, x2, y2):
    A = np.array([x1, y1], dtype=float)
    B = np.array([x2, y2], dtype=float)
    P = np.array([px, py], dtype=float)
    AB = B - A
    ab2 = np.dot(AB, AB)
    if ab2 == 0:
        return float(np.linalg.norm(P - A))
    t = np.clip(np.dot(P - A, AB) / ab2, 0.0, 1.0)
    return float(np.linalg.norm(P - (A + t * AB)))

def frame_to_points(frame):
    rows = []
    for p in frame.get("player_data", []):
        pid = safe_int(p.get("player_id"))
        x = p.get("x")
        y = p.get("y")
        if (x is None or y is None) and isinstance(p.get("coordinates"), dict):
            x = p["coordinates"].get("x")
            y = p["coordinates"].get("y")
        info = player_lookup.get(pid, {})
        rows.append({
            "frame"      : frame.get("frame"),
            "timestamp"  : frame.get("timestamp"),
            "period"     : frame.get("period"),
            "player_id"  : pid,
            "player_name": info.get("name", str(pid)),
            "team_id"    : info.get("team_id"),
            "team_name"  : team_lookup.get(info.get("team_id"), str(info.get("team_id"))),
            "position"   : info.get("position", ""),
            "x"          : x,
            "y"          : y,
            "is_detected": p.get("is_detected"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.dropna(subset=["x", "y"]).reset_index(drop=True)
    return df

def team_attack_direction(team_id, period):
    sides = meta.get("home_team_side", [])
    if period is None or period < 1 or period > len(sides):
        return None
    home_side = sides[period - 1]
    if team_id == home_team_id:
        return +1 if home_side == "left_to_right" else -1
    if team_id == away_team_id:
        return -1 if home_side == "left_to_right" else +1
    return None

def distance_to_nearest_boundary(player, pitch_length=104, pitch_width=68):
    x, y = player["x"], player["y"]
    half_l, half_w = pitch_length / 2, pitch_width / 2
    return min(
        abs(x + half_l),
        abs(half_l - x),
        abs(y + half_w),
        abs(half_w - y),
    )

In [8]:
def nearest_defender_distance(ball_carrier, defenders):
    if defenders.empty:
        return np.nan
    dists = defenders.apply(
        lambda r: euclidean(ball_carrier["x"], ball_carrier["y"], r["x"], r["y"]), axis=1
    )
    return float(dists.min())

def second_nearest_defender_distance(ball_carrier, defenders):
    if len(defenders) < 2:
        return np.nan
    dists = defenders.apply(
        lambda r: euclidean(ball_carrier["x"], ball_carrier["y"], r["x"], r["y"]), axis=1
    ).sort_values()
    return float(dists.iloc[1])

def count_defenders_within_radius(ball_carrier, defenders, radius):
    if defenders.empty:
        return 0
    dists = defenders.apply(
        lambda r: euclidean(ball_carrier["x"], ball_carrier["y"], r["x"], r["y"]), axis=1
    )
    return int((dists <= radius).sum())

def pressure_score_rule(ball_carrier, defenders, pitch_length=104, pitch_width=68):
    nd = nearest_defender_distance(ball_carrier, defenders)
    nd2 = second_nearest_defender_distance(ball_carrier, defenders)
    n3 = count_defenders_within_radius(ball_carrier, defenders, 3.0)
    n5 = count_defenders_within_radius(ball_carrier, defenders, 5.0)
    boundary_dist = distance_to_nearest_boundary(
        ball_carrier, pitch_length=pitch_length, pitch_width=pitch_width
    )
    if np.isnan(nd):
        return {
            "pressure_label"              : 1,
            "nearest_defender_dist"       : np.nan,
            "second_nearest_defender_dist": np.nan,
            "defenders_within_3m"         : 0,
            "defenders_within_5m"         : 0,
            "boundary_dist"               : boundary_dist,
            "notes"                       : "No nearby defenders detected",
        }
    score = 0.0
    notes = []
    if nd < 1.5:
        score += 2.0;  notes.append(f"nearest defender very close ({nd:.2f}m)")
    elif nd < 3.0:
        score += 1.5;  notes.append(f"nearest defender close ({nd:.2f}m)")
    elif nd < 5.0:
        score += 0.75; notes.append(f"nearest defender moderately close ({nd:.2f}m)")
    if not np.isnan(nd2):
        if nd2 < 3.0:
            score += 1.25; notes.append(f"second defender close ({nd2:.2f}m)")
        elif nd2 < 5.0:
            score += 0.5;  notes.append(f"second defender in support ({nd2:.2f}m)")
    if n3 >= 2:
        score += 1.5;  notes.append(f"{n3} defenders within 3m")
    elif n5 >= 2:
        score += 0.75; notes.append(f"{n5} defenders within 5m")
    if boundary_dist < 1.5:
        score += 1.0;  notes.append(f"very close to boundary ({boundary_dist:.2f}m)")
    elif boundary_dist < 3.0:
        score += 0.5;  notes.append(f"close to boundary ({boundary_dist:.2f}m)")
    label = 3 if score >= 3.0 else (2 if score >= 1.5 else 1)
    return {
        "pressure_label"              : label,
        "nearest_defender_dist"       : nd,
        "second_nearest_defender_dist": nd2,
        "defenders_within_3m"         : n3,
        "defenders_within_5m"         : n5,
        "boundary_dist"               : boundary_dist,
        "notes"                       : "; ".join(notes) if notes else "Low pressure",
    }

In [9]:
def draw_centered_pitch(ax, pitch_length=104, pitch_width=68):
    hl, hw = pitch_length / 2, pitch_width / 2
    for x in (-hl, hl):
        ax.plot([x, x], [-hw, hw], lw=1.5, color="black")
    for y in (-hw, hw):
        ax.plot([-hl, hl], [y, y], lw=1.5, color="black")
    ax.plot([0, 0], [-hw, hw], lw=1.5, color="purple", alpha=0.5)
    ax.add_patch(Circle((0, 0), 9.15, fill=False, lw=1.5, color="black"))
    ax.add_patch(Circle((0, 0), 0.3,  color="black"))
    pd_, pw = 16.5, 40.32
    sd, sw  = 5.5,  18.32
    for sign in (-1, 1):
        base_x = -hl if sign == -1 else hl - pd_
        ax.add_patch(Rectangle((base_x, -pw/2), pd_, pw, fill=False, lw=1.5, color="black"))
        base_x = -hl if sign == -1 else hl - sd
        ax.add_patch(Rectangle((base_x, -sw/2), sd,  sw, fill=False, lw=1.5, color="black"))
    ax.add_patch(Circle((-hl + 11, 0), 0.3, color="black"))
    ax.add_patch(Circle(( hl - 11, 0), 0.3, color="black"))
    ax.add_patch(Arc((-hl + 11, 0), 18.3, 18.3, angle=0, theta1=310, theta2=50,  lw=1.5, color="black"))
    ax.add_patch(Arc(( hl - 11, 0), 18.3, 18.3, angle=0, theta1=130, theta2=230, lw=1.5, color="black"))
    ax.set_xlim(-hl - 2, hl + 2)
    ax.set_ylim(-hw - 2, hw + 2)
    ax.set_aspect("equal")
    ax.axis("off")

In [10]:
def count_lane_blockers_simple(ball_carrier, receiver, defenders, lane_radius=2.0):
    if defenders.empty:
        return 0
    x1, y1 = ball_carrier["x"], ball_carrier["y"]
    x2, y2 = receiver["x"], receiver["y"]
    count = 0
    for _, d in defenders.iterrows():
        dist = point_to_segment_distance(d["x"], d["y"], x1, y1, x2, y2)
        if dist <= lane_radius:
            count += 1
    return count

def nearest_defender_to_player(player_row, defenders):
    if defenders.empty:
        return np.nan
    dists = defenders.apply(
        lambda r: euclidean(player_row["x"], player_row["y"], r["x"], r["y"]),
        axis=1
    )
    return float(dists.min())

In [11]:
def attacker_receive_weight(ball_carrier, receiver, defenders):
    pass_dist = euclidean(
        ball_carrier["x"], ball_carrier["y"],
        receiver["x"], receiver["y"]
    )
    lane_blockers = count_lane_blockers_simple(ball_carrier, receiver, defenders, lane_radius=2.0)
    recv_def_dist = nearest_defender_to_player(receiver, defenders)
    weight = 1.0
    # Distance effect
    if pass_dist <= 12:
        weight *= 1.15
    elif pass_dist <= 22:
        weight *= 1.0
    elif pass_dist <= 32:
        weight *= 0.8
    else:
        weight *= 0.6
    # Lane blockage effect
    if lane_blockers == 0:
        weight *= 1.15
    elif lane_blockers == 1:
        weight *= 0.95
    elif lane_blockers == 2:
        weight *= 0.75
    else:
        weight *= 0.5
    # Defender pressure near receiver
    if not np.isnan(recv_def_dist):
        if recv_def_dist > 5:
            weight *= 1.15
        elif recv_def_dist > 3:
            weight *= 1.0
        elif recv_def_dist > 2:
            weight *= 0.8
        else:
            weight *= 0.6
    return max(0.15, float(weight))

In [12]:
def get_player_row_in_frame(frame_idx, player_id):
    if frame_idx < 0 or frame_idx >= len(frames):
        return None
    pts = frame_to_points(frames[frame_idx])
    if pts.empty:
        return None
    row = pts[pts["player_id"] == player_id]
    if row.empty:
        return None
    return row.iloc[0]

In [13]:
def estimate_player_orientation(frame_idx, player_id, window=1, min_movement=0.25):
    prev_row = get_player_row_in_frame(frame_idx - window, player_id)
    curr_row = get_player_row_in_frame(frame_idx, player_id)
    next_row = get_player_row_in_frame(frame_idx + window, player_id)
    if curr_row is None:
        return None
    if prev_row is not None and next_row is not None:
        dx = next_row["x"] - prev_row["x"]
        dy = next_row["y"] - prev_row["y"]
    elif next_row is not None:
        dx = next_row["x"] - curr_row["x"]
        dy = next_row["y"] - curr_row["y"]
    elif prev_row is not None:
        dx = curr_row["x"] - prev_row["x"]
        dy = curr_row["y"] - prev_row["y"]
    else:
        return None
    mag = np.sqrt(dx**2 + dy**2)
    if mag < min_movement:
        return None
    ux = dx / mag
    uy = dy / mag
    return {
        "ux": float(ux),
        "uy": float(uy),
        "dx": float(dx),
        "dy": float(dy),
        "movement_mag": float(mag),
        "angle_deg": float(np.degrees(np.arctan2(uy, ux)))
    }

In [14]:
def get_player_orientation_or_fallback(frame_idx, player_row, default_x=1.0, default_y=0.0):
    pid = safe_int(player_row["player_id"])
    orient = estimate_player_orientation(
        frame_idx=frame_idx,
        player_id=pid,
        window=1,
        min_movement=0.25
    )
    if orient is not None:
        return orient["ux"], orient["uy"], "estimated"
    return float(default_x), float(default_y), "fallback"

In [15]:
def directional_gaussian_influence(
    X, Y,
    px, py,
    ux, uy,
    sigma=4.5,
    weight=1.0,
    front_boost=1.6,
    back_scale=0.45
):
    dx = X - px
    dy = Y - py
    base = np.exp(-((dx**2 + dy**2) / (2 * sigma**2)))
    mag = np.sqrt(dx**2 + dy**2) + 1e-8
    vx = dx / mag
    vy = dy / mag
    alignment = ux * vx + uy * vy
    alignment = np.clip(alignment, -1.0, 1.0)
    orient_scale = back_scale + ((alignment + 1.0) / 2.0) * (front_boost - back_scale)
    return weight * base * orient_scale

In [ ]:
BALL_IN_AIR_Z_THRESHOLD = 0.6
CLEAR_POSSESSION_DIST = 1.6
MAX_POSSESSION_DIST = 2.8
DUEL_DISTANCE_GAP = 0.5
CROWDED_RADIUS = 2.5
CROWD_THRESHOLD = 3

def get_ball_xyz(frame):
    ball = frame.get("ball_data", {})
    if not isinstance(ball, dict):
        return None, None, None
    return ball.get("x"), ball.get("y"), ball.get("z")

In [18]:
def get_ball_candidates(frame, max_ball_distance=MAX_POSSESSION_DIST, top_n=4):
    pts = frame_to_points(frame)
    if pts.empty:
        return pd.DataFrame()
    bx, by, bz = get_ball_xyz(frame)
    if bx is None or by is None:
        return pd.DataFrame()
    pts = pts.copy()
    pts["dist_to_ball"] = pts.apply(
        lambda r: euclidean(bx, by, r["x"], r["y"]), axis=1
    )
    pts = pts.sort_values("dist_to_ball").reset_index(drop=True)
    return pts[pts["dist_to_ball"] <= max_ball_distance].head(top_n).copy()

def get_ball_candidates_by_idx(frame_idx, max_ball_distance=MAX_POSSESSION_DIST, top_n=4):
    if frame_idx < 0 or frame_idx >= len(frames):
        return None, pd.DataFrame()
    frame = frames[frame_idx]
    return frame, get_ball_candidates(frame, max_ball_distance=max_ball_distance, top_n=top_n)

def count_players_near_ball(candidates, radius=CROWDED_RADIUS):
    if candidates.empty:
        return 0
    return int((candidates["dist_to_ball"] <= radius).sum())

def _possession_confidence(candidates, top1, top2, d1, d2, prev_carrier_id):
    p1 = safe_int(top1["player_id"])
    t1 = safe_int(top1["team_id"])
    t2 = safe_int(top2["team_id"]) if top2 is not None else None
    distance_gap = d2 - d1
    crowded_count = count_players_near_ball(candidates, radius=CROWDED_RADIUS)
    confidence = 0.0
    reasons = []
    if d1 <= CLEAR_POSSESSION_DIST:
        confidence += 0.45
        reasons.append(f"nearest player very close ({d1:.2f}m)")
    elif d1 <= MAX_POSSESSION_DIST:
        confidence += 0.20
        reasons.append(f"nearest player somewhat close ({d1:.2f}m)")
    if np.isfinite(d2):
        if distance_gap >= 0.75:
            confidence += 0.30
            reasons.append(f"clear gap to second player ({distance_gap:.2f}m)")
        elif distance_gap >= DUEL_DISTANCE_GAP:
            confidence += 0.15
            reasons.append(f"moderate gap to second player ({distance_gap:.2f}m)")
        else:
            confidence -= 0.25
            reasons.append(f"small gap to second player ({distance_gap:.2f}m)")
    if prev_carrier_id is not None and prev_carrier_id == p1:
        confidence += 0.20
        reasons.append("same as previous likely carrier")
    if crowded_count >= CROWD_THRESHOLD:
        confidence -= 0.20
        reasons.append(f"crowded ball zone ({crowded_count} players within {CROWDED_RADIUS}m)")
    if top2 is not None and t1 is not None and t2 is not None:
        if t1 != t2 and distance_gap < DUEL_DISTANCE_GAP:
            confidence -= 0.25
            reasons.append("possible duel / second-ball moment")
    return max(0.0, min(1.0, confidence)), reasons

def classify_possession_from_frame(frame, prev_carrier_id=None,
                                   max_ball_distance=MAX_POSSESSION_DIST):
    bx, by, bz = get_ball_xyz(frame)
    candidates = get_ball_candidates(frame, max_ball_distance=max_ball_distance, top_n=4)
    _no_poss = lambda reason: {
        "possession_state": "no_possession",
        "ball_carrier_id": None, "team_id": None,
        "carrier_x": None, "carrier_y": None,
        "dist_to_ball": None, "confidence": 0.0, "reason": reason
    }
    if candidates.empty:
        return _no_poss("no player close enough to ball")
    if bz is not None and bz > BALL_IN_AIR_Z_THRESHOLD:
        return _no_poss(f"ball in air (z={bz:.2f})")
    top1 = candidates.iloc[0]
    top2 = candidates.iloc[1] if len(candidates) > 1 else None
    d1 = float(top1["dist_to_ball"])
    d2 = float(top2["dist_to_ball"]) if top2 is not None else np.inf
    p1 = safe_int(top1["player_id"])
    t1 = safe_int(top1["team_id"])
    confidence, reasons = _possession_confidence(
        candidates, top1, top2, d1, d2, prev_carrier_id
    )
    if confidence >= 0.55:
        return {
            "possession_state": "clear_possession",
            "ball_carrier_id": p1, "team_id": t1,
            "carrier_x": float(top1["x"]), "carrier_y": float(top1["y"]),
            "dist_to_ball": d1,
            "confidence": confidence, "reason": "; ".join(reasons)
        }
    elif confidence >= 0.25:
        return {
            "possession_state": "contested",
            "ball_carrier_id": None, "team_id": None,
            "carrier_x": None, "carrier_y": None,
            "dist_to_ball": None,
            "confidence": confidence, "reason": "; ".join(reasons)
        }
    return _no_poss("; ".join(reasons))

def classify_possession(frame_idx, max_ball_distance=MAX_POSSESSION_DIST):
    if frame_idx < 0 or frame_idx >= len(frames):
        return {
            "possession_state": "no_possession", "ball_carrier": None,
            "candidates": pd.DataFrame(), "confidence": 0.0, "reason": "invalid frame"
        }
    frame = frames[frame_idx]
    prev_pid = None
    if frame_idx > 0:
        prev_cands = get_ball_candidates(frames[frame_idx - 1],
                                         max_ball_distance=max_ball_distance, top_n=3)
        if not prev_cands.empty and prev_cands.iloc[0]["dist_to_ball"] <= CLEAR_POSSESSION_DIST:
            prev_pid = safe_int(prev_cands.iloc[0]["player_id"])
    info = classify_possession_from_frame(frame, prev_carrier_id=prev_pid,
                                          max_ball_distance=max_ball_distance)
    candidates = get_ball_candidates(frame, max_ball_distance=max_ball_distance, top_n=4)
    ball_carrier = None
    if info["possession_state"] == "clear_possession" and not candidates.empty:
        ball_carrier = candidates.iloc[0]
    return {
        "possession_state": info["possession_state"],
        "ball_carrier": ball_carrier,
        "candidates": candidates,
        "confidence": info["confidence"],
        "reason": info["reason"]
    }

In [16]:
def build_blended_control_map(
    frame_idx,
    grid_step=1.5,
    max_ball_distance=2.5,
    att_sigma=4.5,
    def_sigma=5.0
):
    if frame_idx < 0 or frame_idx >= len(frames):
        return None
    frame = frames[frame_idx]
    pts = frame_to_points(frame)
    if pts.empty:
        return None
    bx, by, bz = get_ball_xyz(frame)
    possession_info = classify_possession(frame_idx, max_ball_distance=max_ball_distance)
    ball_carrier = possession_info["ball_carrier"]
    ball_info = {
        "x": bx, "y": by, "z": bz,
        "possession_state": possession_info["possession_state"],
        "possession_confidence": possession_info["confidence"],
        "possession_reason": possession_info["reason"]
    }
    # No possession case
    if ball_carrier is None or pd.isna(ball_carrier["team_id"]):
        return {
            "frame": frame,
            "pts": pts,
            "ball_carrier": None,
            "ball_info": ball_info,
            "receivers": pd.DataFrame(),
            "defenders": pd.DataFrame(),
            "X": None,
            "Y": None,
            "attacker_field": None,
            "defender_field": None,
            "contested": None,
            "balance": None,
            "carrier_team_id": None
        }
    carrier_team_id = safe_int(ball_carrier["team_id"])
    # Ball carrier's teammates = receivers
    receivers = pts[
        (pts["team_id"] == carrier_team_id) &
        (pts["player_id"] != ball_carrier["player_id"])
    ].copy()
    defenders = pts[pts["team_id"] != carrier_team_id].copy()
    half_l = pitch_length / 2
    half_w = pitch_width / 2
    xs = np.arange(-half_l, half_l + grid_step, grid_step)
    ys = np.arange(-half_w, half_w + grid_step, grid_step)
    X, Y = np.meshgrid(xs, ys)
    attacker_field = np.zeros_like(X, dtype=float)
    defender_field = np.zeros_like(X, dtype=float)
    attack_dir = team_attack_direction(carrier_team_id, frame.get("period"))
    def_fallback_x = -float(attack_dir) if attack_dir is not None else -1.0
    # Receivers: directional blue influence
    for _, a in receivers.iterrows():
        w = attacker_receive_weight(ball_carrier, a, defenders)
        ux, uy, orient_source = get_player_orientation_or_fallback(
            frame_idx=frame_idx,
            player_row=a,
            default_x=float(attack_dir) if attack_dir is not None else 1.0,
            default_y=0.0
        )
        attacker_field += directional_gaussian_influence(
            X, Y,
            a["x"], a["y"],
            ux, uy,
            sigma=att_sigma,
            weight=w,
            front_boost=1.6,
            back_scale=0.45
        )
    # Defenders: directional red influence
    for _, d in defenders.iterrows():
        ux, uy, orient_source = get_player_orientation_or_fallback(
            frame_idx=frame_idx,
            player_row=d,
            default_x=def_fallback_x,
            default_y=0.0
        )
        defender_field += directional_gaussian_influence(
            X, Y,
            d["x"], d["y"],
            ux, uy,
            sigma=def_sigma,
            weight=1.0,
            front_boost=1.5,
            back_scale=0.55
        )
    if attacker_field.max() > 0:
        attacker_norm = attacker_field / attacker_field.max()
    else:
        attacker_norm = attacker_field.copy()
    if defender_field.max() > 0:
        defender_norm = defender_field / defender_field.max()
    else:
        defender_norm = defender_field.copy()
    contested = np.minimum(attacker_norm, defender_norm)
    balance = attacker_norm / (attacker_norm + defender_norm + 1e-8)
    return {
        "frame": frame,
        "pts": pts,
        "ball_carrier": ball_carrier,
        "ball_info": ball_info,
        "receivers": receivers,
        "defenders": defenders,
        "X": X,
        "Y": Y,
        "attacker_field": attacker_norm,
        "defender_field": defender_norm,
        "contested": contested,
        "balance": balance,
        "carrier_team_id": carrier_team_id
    }

In [19]:
def plot_blended_control_map(
    frame_idx,
    grid_step=1.5,
    max_ball_distance=MAX_POSSESSION_DIST,
    att_sigma=4.5,
    def_sigma=5.0,
    show_labels=True,
    show_balance_line=True,
    show_pressure=True,
    show_receiver_orientation=False,
    save_path=None,
    show_plot=True
):
    result = build_blended_control_map(
        frame_idx=frame_idx,
        grid_step=grid_step,
        max_ball_distance=max_ball_distance,
        att_sigma=att_sigma,
        def_sigma=def_sigma
    )
    if result is None:
        print(f"Could not build control map for frame {frame_idx}.")
        return None
    frame = result["frame"]
    pts = result["pts"]
    ball_carrier = result["ball_carrier"]
    ball_info = result["ball_info"]
    receivers = result["receivers"]
    defenders = result["defenders"]
    X = result["X"]
    Y = result["Y"]
    attacker_field = result["attacker_field"]
    defender_field = result["defender_field"]
    contested = result["contested"]
    balance = result["balance"]
    fig, ax = plt.subplots(figsize=(13, 8))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    draw_centered_pitch(ax, pitch_length=pitch_length, pitch_width=pitch_width)
    if ball_carrier is not None and attacker_field is not None and defender_field is not None:
        attacker_plot = attacker_field.copy()
        defender_plot = defender_field.copy()
        attacker_plot[attacker_plot < 0.08] = 0.0
        defender_plot[defender_plot < 0.08] = 0.0
        rgb = np.ones((attacker_plot.shape[0], attacker_plot.shape[1], 3), dtype=float)
        rgb[..., 1] -= defender_plot * 0.6
        rgb[..., 2] -= defender_plot * 0.6
        rgb[..., 0] -= attacker_plot * 0.6
        rgb[..., 1] -= attacker_plot * 0.6
        rgb = np.clip(rgb, 0, 1)
        alpha = np.maximum(attacker_plot, defender_plot)
        alpha[alpha < 0.05] = 0.0
        alpha = np.clip(alpha * 0.85, 0, 0.85)
        ax.imshow(
            rgb,
            extent=[-pitch_length/2, pitch_length/2, -pitch_width/2, pitch_width/2],
            origin="lower",
            alpha=alpha,
            aspect="auto",
            zorder=0
        )
        ax.contour(
            X, Y, contested,
            levels=[0.35, 0.55],
            colors=["purple", "purple"],
            linewidths=[1.0, 1.5],
            zorder=1
        )
        if show_balance_line:
            ax.contour(
                X, Y, balance,
                levels=[0.5],
                colors="black",
                linewidths=1.0,
                linestyles="--",
                zorder=1
            )
    home_players = pts[pts["team_id"] == home_team_id]
    away_players = pts[pts["team_id"] == away_team_id]
    unknown_players = pts[pts["team_id"].isna()]
    if not home_players.empty:
        ax.scatter(
            home_players["x"], home_players["y"],
            s=110, label=home_team_name, zorder=3
        )
    if not away_players.empty:
        ax.scatter(
            away_players["x"], away_players["y"],
            s=110, marker="s", label=away_team_name, zorder=3
        )
    if not unknown_players.empty:
        ax.scatter(
            unknown_players["x"], unknown_players["y"],
            s=110, marker="x", label="Unknown team", zorder=3
        )
    if ball_carrier is not None:
        ax.scatter(
            [ball_carrier["x"]], [ball_carrier["y"]],
            s=260,
            facecolors="none",
            edgecolors="yellow",
            linewidths=2.5,
            label="Ball Carrier",
            zorder=4
        )
    if ball_info is not None:
        ax.scatter(
            [ball_info["x"]], [ball_info["y"]],
            s=160,
            marker="o",
            edgecolors="black",
            label="Ball",
            zorder=4
        )
    if show_receiver_orientation and ball_carrier is not None:
        if not receivers.empty:
            for _, r in receivers.iterrows():
                pid = safe_int(r["player_id"])
                if pid is None:
                    continue
                orient = estimate_player_orientation(
                    frame_idx=frame_idx,
                    player_id=pid,
                    window=1,
                    min_movement=0.25
                )
                if orient is None:
                    continue
                arrow_len = 2.5
                dx = orient["ux"] * arrow_len
                dy = orient["uy"] * arrow_len
                ax.arrow(
                    r["x"], r["y"],
                    dx, dy,
                    head_width=0.6,
                    head_length=0.8,
                    fc="black",
                    ec="black",
                    linewidth=1.2,
                    length_includes_head=True,
                    zorder=5
                )
        if not defenders.empty:
            for _, d in defenders.iterrows():
                pid = safe_int(d["player_id"])
                if pid is None:
                    continue
                orient = estimate_player_orientation(
                    frame_idx=frame_idx,
                    player_id=pid,
                    window=1,
                    min_movement=0.25
                )
                if orient is None:
                    continue
                arrow_len = 2.5
                dx = orient["ux"] * arrow_len
                dy = orient["uy"] * arrow_len
                ax.arrow(
                    d["x"], d["y"],
                    dx, dy,
                    head_width=0.6,
                    head_length=0.8,
                    fc="dimgray",
                    ec="dimgray",
                    linewidth=1.0,
                    length_includes_head=True,
                    zorder=5
                )
    if show_labels:
        for _, r in pts.iterrows():
            ax.text(
                r["x"], r["y"] + 0.8,
                str(r["player_id"]),
                fontsize=7,
                ha="center",
                va="bottom",
                zorder=6
            )
    possession_text = "Possession: N/A"
    pressure_text = "Pressure: N/A"
    if ball_info is not None:
        possession_text = (
            f"Possession: {ball_info.get('possession_state', 'N/A')} | "
            f"confidence: {ball_info.get('possession_confidence', 0):.2f}"
        )
    if show_pressure and ball_carrier is not None and defenders is not None:
        pressure_info = pressure_score_rule(
            ball_carrier,
            defenders,
            pitch_length=pitch_length,
            pitch_width=pitch_width
        )
        if pd.notna(pressure_info["nearest_defender_dist"]):
            pressure_text = (
                f"Pressure: {pressure_info['pressure_label']} | "
                f"nearest defender: {pressure_info['nearest_defender_dist']:.2f}m"
            )
        else:
            pressure_text = f"Pressure: {pressure_info['pressure_label']}"
    ax.text(
        1.02, 0.95,
        possession_text,
        transform=ax.transAxes,
        fontsize=10,
        va="top"
    )
    ax.text(
        1.02, 0.90,
        pressure_text,
        transform=ax.transAxes,
        fontsize=10,
        va="top"
    )
    if ball_info is not None:
        ax.text(
            1.02, 0.82,
            f"Reason: {ball_info.get('possession_reason', '')}",
            transform=ax.transAxes,
            fontsize=8,
            va="top",
            wrap=True
        )
    carrier_id = safe_int(ball_carrier["player_id"]) if ball_carrier is not None else "None"
    ax.set_title(
        f"Blended Receive vs Defender Control | frame={frame.get('frame')} | "
        f"t={frame.get('timestamp')} | carrier={carrier_id}"
    )
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="upper right")
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight", facecolor="white")
    if show_plot:
        plt.show()
    else:
        plt.close(fig)
    return result

In [20]:
def build_possession_table(frames, min_frame_id=27, max_ball_distance=MAX_POSSESSION_DIST):
    rows = []
    prev_carrier_id = None
    for frame in frames:
        frame_id = frame.get("frame")
        if frame_id is None or frame_id < min_frame_id:
            continue
        bx, by, bz = get_ball_xyz(frame)
        info = classify_possession_from_frame(
            frame,
            prev_carrier_id=prev_carrier_id,
            max_ball_distance=max_ball_distance
        )
        row = {
            "match_id": MATCH_ID,
            "frame_id": frame.get("frame"),
            "timestamp": frame.get("timestamp"),
            "period": frame.get("period"),
            "ball_x": bx,
            "ball_y": by,
            "ball_z": bz,
            **info
        }
        rows.append(row)
        if info["possession_state"] == "clear_possession":
            prev_carrier_id = info["ball_carrier_id"]
        else:
            prev_carrier_id = None
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["period", "frame_id"]).reset_index(drop=True)
    return df

In [21]:
possession_df = build_possession_table(frames, min_frame_id=27, max_ball_distance=MAX_POSSESSION_DIST)

print(f"Possession rows: {len(possession_df):,}")
print(possession_df.head())
print(possession_df["possession_state"].value_counts(dropna=False))

Possession rows: 59,034
  match_id  frame_id    timestamp  period  ball_x  ball_y  ball_z  \
0  1886347        27  00:00:01.70     1.0    0.52   -0.37    0.30   
1  1886347        28  00:00:01.80     1.0    0.03   -0.36    0.33   
2  1886347        29  00:00:01.90     1.0   -1.03   -0.23    0.34   
3  1886347        30  00:00:02.00     1.0   -2.58   -0.12    0.35   
4  1886347        31  00:00:02.10     1.0   -4.36   -0.01    0.33   

  possession_state  ball_carrier_id  team_id  carrier_x  carrier_y  \
0        contested              NaN      NaN        NaN        NaN   
1        contested              NaN      NaN        NaN        NaN   
2    no_possession              NaN      NaN        NaN        NaN   
3    no_possession              NaN      NaN        NaN        NaN   
4    no_possession              NaN      NaN        NaN        NaN   

   dist_to_ball  confidence                                 reason  
0           NaN        0.45      nearest player very close (0.88m)  
1 

In [22]:
clear_possession_frame_ids = possession_df.loc[
    possession_df["possession_state"] == "clear_possession", "frame_id"
].dropna().astype(int).tolist()
clear_possession_frame_indices = [
    i for i, fr in enumerate(frames)
    if fr.get("frame") in set(clear_possession_frame_ids)
]
default_frame_idx = clear_possession_frame_indices[0] if len(clear_possession_frame_indices) > 0 else 0

slider = widgets.IntSlider(
    value=default_frame_idx,
    min=0,
    max=len(frames) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False
)
show_labels_cb = widgets.Checkbox(value=True, description="Show player IDs")
show_balance_cb = widgets.Checkbox(value=True, description="Show 50/50 line")
show_pressure_cb = widgets.Checkbox(value=True, description="Show pressure")
show_orientation_cb = widgets.Checkbox(value=False, description="Show player orientation")
show_clear_only_cb = widgets.Checkbox(value=False, description="Clear possession only")

def update_slider_range(change):
    if show_clear_only_cb.value and len(clear_possession_frame_indices) > 0:
        slider.min = 0
        slider.max = len(clear_possession_frame_indices) - 1
        slider.step = 1
        slider.description = "Poss idx:"
        slider.value = min(slider.value, slider.max)
    else:
        slider.min = 0
        slider.max = len(frames) - 1
        slider.step = 1
        slider.description = "Frame idx:"
        slider.value = min(slider.value, slider.max)
show_clear_only_cb.observe(update_slider_range, names="value")

def update_control_map(frame_idx, show_labels, show_balance_line, show_pressure,
                       show_player_orientation, clear_possession_only):
    plt.close("all")
    if clear_possession_only and len(clear_possession_frame_indices) > 0:
        actual_frame_idx = clear_possession_frame_indices[frame_idx]
    else:
        actual_frame_idx = frame_idx
    plot_blended_control_map(
        frame_idx=actual_frame_idx,
        show_labels=show_labels,
        show_balance_line=show_balance_line,
        show_pressure=show_pressure,
        show_receiver_orientation=show_player_orientation
    )
ui = widgets.VBox([
    slider,
    show_labels_cb,
    show_balance_cb,
    show_pressure_cb,
    show_orientation_cb,
    show_clear_only_cb
])
out = widgets.interactive_output(
    update_control_map,
    {
        "frame_idx": slider,
        "show_labels": show_labels_cb,
        "show_balance_line": show_balance_cb,
        "show_pressure": show_pressure_cb,
        "show_player_orientation": show_orientation_cb,
        "clear_possession_only": show_clear_only_cb
    }
)
display(ui, out)

Output()